WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

# Contact-free magnetic catheter steering by target following

The catheter leaves a guide sheath at the vessel entry and is advanced in `K` insertion
steps. The sheath is modeled by clamping every node that has not yet left it, so at
depth `d_k` the free rod is the distal material of length `d_k`. The wall is not
modeled, so there is no contact anywhere.

- **Design variables.** `(polar, azimuthal, strength)` of the remnant flux of every
  fiber of every element, and of the applied flux at a set of control depths that is
  interpolated to every insertion step. Both magnitudes are bounded by their extents.
- **Objective.** `J = log(E / E_0)`, where `E = Σ_k Σ_n ‖x_n^k − p(σ_n^k)‖²` compares
  every node with the target path at its path coordinate `σ_n^k = s_n − L + d_k`, and
  `E_0` is the same sum for a straight unactuated insertion. `J` is zero at the start
  and `exp(J)` is the fraction of the straight-insertion squared error that is left.
- **Constraints.** None beyond the box bounds, so MMA receives one inactive dummy
  constraint, as in `mag_with_obstacle`.
- **Units.** SI throughout, because the hard-magnetic energy divides by `μ0` in SI.

### Imports

In [ ]:
import functools as ft
import pathlib
import time
import typing

import chex
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv
import scipy.interpolate as spy_interp

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.mma as _mma
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz
import crops.tools.geom as _geom

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)

results_dir = pathlib.Path("crops/data/results/mag_catheter")
results_dir.mkdir(parents=True, exist_ok=True)

### Target, material, solver, and baseline settings

In [ ]:
target_file = pathlib.Path("crops/data/targets/catheter_k_002_left_0.csv")
stl_file = pathlib.Path("crops/data/obstacles/catheter_k_002_left_0.csv")
length_scale = 1.0  # csv units to m, use 1e-3 if the csv is in mm
stl_length_scale = 1.0  # stl units to m

# Catheter tube of hard-magnetic elastomer, with the material values of Wang (2025).
outer_radius = 0.5e-3  # [m]
inner_radius = 0.35e-3  # [m]
shear_modulus = 3.03e5  # [Pa]
max_remnant_flux = 0.143  # [T]
max_applied_flux = 2.0e-2  # [T]

# Newton stops at max(threshold |r0|, abs_threshold). A converged state still carries a
# round-off residual of about eps EA |x| / h, which is 1e-14 to 1e-13 N here and grows
# with the insertion, while magnetic nodal moments at mT fields are 1e-9 to 1e-7 N m.
# The absolute floor must sit between the two. Below the round-off floor every
# already-converged step burns max_iter iterations and is reported as unconverged.
nonlinear_settings = {
  "max_iter": 100,
  "threshold": 1e-8,
  "abs_threshold": 1e-11,
  "lam_min": 0.01,
  "line_search_max_iter": 12,
  "line_search_shrink": 0.5,
  "line_search_alpha_min": 1e-6,
  "line_search_armijo_c": 1e-4,
}
linear_settings = {"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-10}
solver_verbose = True  # prints every Newton solve, K lines per forward solve

baseline_settings = {
  "num_elems": 80,
  "num_fibers": 8,
  "steps_per_elem": 2.0,
  "num_field_controls": 20,
  "sheath_fraction": 0.1,
}

### Setup container and builder

In [ ]:
class CatheterSetup(typing.NamedTuple):
  """Stores every object one target-following optimization needs.

  Element and fiber counts, reference coordinates, directors, and section properties
  are read from `rod` rather than duplicated here.

  Attributes:
    rod: `Rod` of the straight catheter lying along the entry tangent `t₀` and ending
      at the entry point `p₀`, so its tip starts at the sheath exit.
    problem: `CosseratProblem` holding the rod, the sheath clamp schedule, and the
      Newton solver. It carries no obstacle, so no contact is computed.
    insertion_depths: Array of shape `(num_steps,)` [m] of the free length `d_k` at
      every insertion step, rising uniformly to the target path length.
    field_interpolation: Array of shape `(num_steps, num_field_controls)` of the
      piecewise linear weights that map the applied flux at the control depths to
      every insertion step.
    node_path_coords: Array of shape `(num_steps, num_nodes)` [m] of the path
      coordinate `σ = s − L + d_k` of every node, which is positive once the node has
      left the sheath.
    target_positions: Array of shape `(num_steps, num_nodes, 3)` [m] of the target of
      every node at every step. It is `p(σ)` for `σ > 0` and the sheath line
      `p₀ + σ t₀` otherwise, so sheathed nodes carry no error.
    target_points: Array of shape `(num_samples, 3)` [m] of the dense target path,
      used for plotting and for the path curvature estimate.
    entry_tangent: Array of shape `(3,)` of the unit tangent `t₀` of the target at the
      entry, which is also the insertion direction.
    origin: Array of shape `(3,)` [m] of the first csv point in the csv frame. Every
      coordinate here is measured from it, so the entry point is the origin, and the
      STL must be shifted by it before any comparison.
    straight_error: Scalar [m²] `E₀`, the squared error of a straight unactuated
      insertion summed over all nodes and steps.
  """

  rod: _mesher.Rod
  problem: _fe.CosseratProblem
  insertion_depths: np.ndarray
  field_interpolation: np.ndarray
  node_path_coords: np.ndarray
  target_positions: jnp.ndarray
  target_points: np.ndarray
  entry_tangent: np.ndarray
  origin: np.ndarray
  straight_error: float


def build_setup(
  num_elems: int,
  num_fibers: int,
  steps_per_elem: float,
  num_field_controls: int,
  sheath_fraction: float,
) -> CatheterSetup:
  """Builds the sheathed catheter, its insertion schedule, and the target.

  The rod starts fully retracted, lying straight along the entry tangent with its tip
  at the entry point. At step `k` every node whose path coordinate is not positive is
  still inside the sheath, so its translation is prescribed as the rigid insertion
  `d_k t₀` and its rotation as zero. All other nodes are free, and only the uniform
  applied field acts on them.

  Algorithm.
    1. Read the target, convert it to m, shift its arc coordinate to start at zero and
       its points to start at the origin, and fit a cubic spline `p(σ)`. Centering
       keeps the node coordinates small, which lowers the round-off floor of the
       residual. Set the path length `ℓ = σ_max`, the entry point `p₀ = p(0) = 0`,
       and the entry tangent `t₀ = p′(0) / ‖p′(0)‖`.
    2. Set the rod length `L = (1 + sheath_fraction) ℓ`, so a sheathed length of
       `sheath_fraction ℓ` remains clamped at full insertion.
    3. Place `num_fibers` fibers on a circle of the mean radius `(r_o + r_i) / 2`,
       each with area `π (r_o² − r_i²) / num_fibers`, and build the straight rod
       `r(s) = p₀ + (s − L) t₀` for `s ∈ [0, L]`. The seed binormal is the Cartesian
       axis least aligned with `t₀`.
    4. Set `K = ⌈steps_per_elem · num_elems / (1 + sheath_fraction)⌉`, so the
       increment `Δ = ℓ / K` is at most `h / steps_per_elem` for the element length
       `h = L / num_elems`. Set `d_k = (k + 1) Δ` and `σ_n^k = s_n − L + d_k`, and clamp
       the nodes with `σ_n^k ≤ 0` at the translation `d_k t₀` and zero rotation. The
       bound on `Δ` matters because the first free node starts from a guess that
       lags the sheath by one increment, so `Δ ≥ h` would invert the exit element.
    5. Build the piecewise linear weights from `num_field_controls` evenly spaced
       control depths in `[d_0, ℓ]` to every `d_k`.
    6. Assemble the Newton solver and the problem without obstacle or self-contact.
    7. Evaluate the node targets `p(σ)` for `σ > 0` and `p₀ + σ t₀` otherwise, and the
       straight-insertion error `E₀ = Σ ‖p₀ + σ t₀ − p̂(σ)‖²`.

  Args:
    num_elems: Number of rod elements, positive. The element length should stay below
      the smallest radius of curvature of the path divided by about ten.
    num_fibers: Number of fibers around the tube wall, positive, typically 8 to 16.
    steps_per_elem: Number of insertion steps per element length, at least one and
      typically two. It sets `K` and keeps the initial guess of each step valid.
    num_field_controls: Number of control depths of the applied flux, positive and at
      most `K`. Fewer controls give a smoother field sequence and fewer variables.
    sheath_fraction: Sheathed length at full insertion as a fraction of the path
      length, in `(0, 1)`. It must exceed one element length over `ℓ`.

  Returns:
    A `CatheterSetup` holding every object one optimization run needs.
  """
  target = pd.read_csv(target_file)
  target_arc = length_scale * (target["s"].values - target["s"].values[0])
  target_xyz = length_scale * target[["x", "y", "z"]].values
  target_curve_fn = spy_interp.CubicSpline(target_arc, target_xyz - target_xyz[0])
  path_length = float(target_arc[-1])
  entry_point = target_curve_fn(0.0)
  entry_tangent = target_curve_fn(0.0, 1)
  entry_tangent = entry_tangent / np.linalg.norm(entry_tangent)
  rod_length = (1.0 + sheath_fraction) * path_length

  mean_radius = 0.5 * (outer_radius + inner_radius)
  fiber_area = np.pi * (outer_radius**2 - inner_radius**2) / num_fibers
  rod = _mesher.build_rod_geometry(
    analytical_curve=lambda arc: (
      jnp.asarray(entry_point) + (arc - rod_length) * jnp.asarray(entry_tangent)
    ),
    parametric_coords=jnp.linspace(0.0, rod_length, num_elems + 1),
    fiber_offsets=jnp.asarray(
      _geom.annular_fiber_offsets(
        num_elems=num_elems, num_fibers=num_fibers, semi_axis_d1=mean_radius
      )
    ),
    fiber_areas=jnp.full((num_elems, num_fibers), fiber_area),
    fiber_mass_density=jnp.full((num_elems, num_fibers), 1100.0),
    fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
    seed_binormal=jnp.asarray(np.eye(3)[np.argmin(np.abs(entry_tangent))]),
  )

  arc_coords = np.asarray(rod.arc_coords)
  num_steps = int(np.ceil(steps_per_elem * num_elems / (1.0 + sheath_fraction)))
  insertion_depths = path_length * np.arange(1, num_steps + 1) / num_steps
  control_depths = np.linspace(insertion_depths[0], path_length, num_field_controls)
  field_interpolation = np.stack(
    [
      np.interp(x=insertion_depths, xp=control_depths, fp=unit)
      for unit in np.eye(num_field_controls)
    ],
    axis=-1,
  )
  node_path_coords = arc_coords[None, :] - rod_length + insertion_depths[:, None]
  sheathed_nodes = [np.flatnonzero(coords <= 0.0) for coords in node_path_coords]
  translations = (_utils.Translation.X, _utils.Translation.Y, _utils.Translation.Z)
  rotations = (
    _utils.Rotation.THETA_X,
    _utils.Rotation.THETA_Y,
    _utils.Rotation.THETA_Z,
  )
  sheath = [
    _bc.DirichletBC(
      values=(
        dof,
        [
          np.full(nodes.size, depth * component)
          for nodes, depth in zip(sheathed_nodes, insertion_depths)
        ],
      ),
      node_targets=sheathed_nodes,
      name=f"sheath_{dof}",
    )
    for dof, component in zip(translations, entry_tangent)
  ] + [
    _bc.DirichletBC(
      values=(dof, [np.zeros(nodes.size) for nodes in sheathed_nodes]),
      node_targets=sheathed_nodes,
      name=f"sheath_{dof}",
    )
    for dof in rotations
  ]
  bc = _bc.process_boundary_conditions(rod=rod, boundary_conditions_list=sheath)

  problem = _fe.CosseratProblem(
    rod=rod,
    bc=bc,
    nr_solver=_solver.NewtonRaphson(
      solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
      nonlinear_settings=nonlinear_settings,
      linear_settings=linear_settings,
      verbose=solver_verbose,
    ),
  )

  on_path = target_curve_fn(np.clip(node_path_coords, 0.0, path_length))
  on_line = entry_point + np.einsum("kn,d->knd", node_path_coords, entry_tangent)
  target_positions = np.where(node_path_coords[..., None] > 0.0, on_path, on_line)
  return CatheterSetup(
    rod=rod,
    problem=problem,
    insertion_depths=insertion_depths,
    field_interpolation=field_interpolation,
    node_path_coords=node_path_coords,
    target_positions=jnp.asarray(target_positions),
    target_points=target_curve_fn(np.linspace(0.0, path_length, 400)),
    entry_tangent=entry_tangent,
    origin=target_xyz[0],
    straight_error=float(np.sum((on_line - target_positions) ** 2)),
  )

### Build and check the magnetic authority

A uniformly magnetized free length `ℓ` under a perpendicular field carries the couple
density `c = A B_r B_a / μ0`, so its clamp curvature is `κ = c ℓ / EI`. If this
full-flux authority is below the largest path curvature, no design can follow the
path and the bounds must be raised first.

In [ ]:
setup = build_setup(**baseline_settings)

path_length = float(setup.insertion_depths[-1])
dense_arc = np.linspace(0.0, path_length, setup.target_points.shape[0])
path_curvature = np.linalg.norm(
  np.gradient(np.gradient(setup.target_points, dense_arc, axis=0), dense_arc, axis=0),
  axis=-1,
)
bending_stiffness = 3.0 * shear_modulus * float(setup.rod.elem_moi_tensor[0, 0, 0])
couple_density = (
  float(setup.rod.elem_areas[0])
  * max_remnant_flux
  * max_applied_flux
  / _utils.PERMEABILITY_VACUUM
)
authority = couple_density * path_length / bending_stiffness

print(f"path length          {1e3 * path_length:.1f} mm")
print(f"rod length           {1e3 * float(setup.rod.arc_coords[-1]):.1f} mm")
print(f"sheathed nodes, end  {int(np.sum(setup.node_path_coords[-1] <= 0.0))}")
print(f"insertion steps      {setup.insertion_depths.shape[0]}")
print(f"min path radius      {1e3 / path_curvature.max():.2f} mm")
print(f"full-flux authority  {authority:.1f} 1/m")
print(f"max path curvature   {path_curvature.max():.1f} 1/m")

### Design variables in polar, azimuthal, and strength form

In [ ]:
@chex.dataclass
class DesignExtents:
  """Stores the physical range of every magnetic design variable.

  Attributes:
    remnant_polar: `Extent` [rad] of the remnant polar angle θ measured from `+z`.
    remnant_azith: `Extent` [rad] of the remnant azimuthal angle φ measured from `+x`.
    remnant_strength: `Extent` [T] of the remnant flux magnitude.
    applied_polar: `Extent` [rad] of the applied polar angle.
    applied_azith: `Extent` [rad] of the applied azimuthal angle.
    applied_strength: `Extent` [T] of the applied flux magnitude.
  """

  remnant_polar: _utils.Extent
  remnant_azith: _utils.Extent
  remnant_strength: _utils.Extent
  applied_polar: _utils.Extent
  applied_azith: _utils.Extent
  applied_strength: _utils.Extent


# Full sphere of directions for both fields. A zero lower strength lets segments end
# unmagnetized, and bounds both magnitudes by construction, so no constraint is needed.
extents = DesignExtents(
  remnant_polar=_utils.Extent(min=0.0, max=np.pi),
  remnant_azith=_utils.Extent(min=-np.pi, max=np.pi),
  remnant_strength=_utils.Extent(min=0.0, max=max_remnant_flux),
  applied_polar=_utils.Extent(min=0.0, max=np.pi),
  applied_azith=_utils.Extent(min=-np.pi, max=np.pi),
  applied_strength=_utils.Extent(min=0.0, max=max_applied_flux),
)


def design_vars_to_opt_vars(
  design_vars: jnp.ndarray,
  extents: DesignExtents,
  setup: CatheterSetup,
) -> tuple[jnp.ndarray, jnp.ndarray]:
  """Expands the normalized MMA design vector into the physical flux fields.

  The design vector is one flat array in `[0, 1]` laid out as

      [ remnant block : 3 num_elems num_fibers ][ applied block : 3 num_controls ].

  Every triple holds `(polar, azimuthal, strength)`, each mapped affinely onto its
  `Extent` and converted by `b = |b| (sin θ cos φ, sin θ sin φ, cos θ)`. The remnant
  flux is a per fiber field in the reference global frame. The applied flux is
  defined at the control depths and interpolated to every insertion step as
  `b_a(d_k) = Σ_c N_kc b_a,c`, a convex combination, so its magnitude never exceeds
  the largest control magnitude.

  Args:
    design_vars: Array of shape `(3 num_elems num_fibers + 3 num_controls,)` of design
      variables in `[0, 1]`.
    extents: `DesignExtents` supplying the physical range of each variable.
    setup: `CatheterSetup` supplying the rod and the field interpolation.

  Returns:
    A tuple `(remnant_mag_flux, applied_mag_flux)` of shapes
      `(num_elems, num_fibers, num_dim=3)` [T] and `(num_steps, num_dim=3)` [T].
  """
  num_elems = setup.rod.fibers.num_elems
  num_fibers = setup.rod.fibers.num_fibers
  num_remnant_vars = 3 * num_elems * num_fibers
  remnant_block = design_vars[:num_remnant_vars].reshape(num_elems, num_fibers, 3)
  applied_block = design_vars[num_remnant_vars:].reshape(-1, 3)

  remnant_mag_flux = _utils.polar_azith_to_vector(
    extents.remnant_polar.renormalize_array(remnant_block[..., 0]),
    extents.remnant_azith.renormalize_array(remnant_block[..., 1]),
    extents.remnant_strength.renormalize_array(remnant_block[..., 2]),
  )
  control_mag_flux = _utils.polar_azith_to_vector(
    extents.applied_polar.renormalize_array(applied_block[:, 0]),
    extents.applied_azith.renormalize_array(applied_block[:, 1]),
    extents.applied_strength.renormalize_array(applied_block[:, 2]),
  )
  applied_mag_flux = jnp.einsum(
    "kc,cd->kd", setup.field_interpolation, control_mag_flux
  )
  return remnant_mag_flux, applied_mag_flux


def opt_vars_to_design_vars(
  remnant_mag_flux: jnp.ndarray,
  control_mag_flux: jnp.ndarray,
  extents: DesignExtents,
) -> jnp.ndarray:
  """Collapses physical flux fields into the normalized MMA design vector.

  This is the inverse of `design_vars_to_opt_vars` at the control depths. Each vector
  is decomposed into `(θ, φ, |b|)` by `vector_to_polar_azith` and normalized by its
  `Extent`. Values outside an extent are left unclipped, so an out of range guess
  stays visible.

  Args:
    remnant_mag_flux: Array of shape `(num_elems, num_fibers, num_dim=3)` [T] of the
      remnant flux in the reference global frame.
    control_mag_flux: Array of shape `(num_controls, num_dim=3)` [T] of the applied
      flux at the control depths.
    extents: `DesignExtents` supplying the physical range of each variable.

  Returns:
    Array of shape `(num_design_var,)` of normalized design variables.
  """
  remnant_polar, remnant_azith, remnant_strength = _utils.vector_to_polar_azith(
    remnant_mag_flux
  )
  applied_polar, applied_azith, applied_strength = _utils.vector_to_polar_azith(
    control_mag_flux
  )
  remnant_block = jnp.stack(
    [
      extents.remnant_polar.normalize_array(remnant_polar),
      extents.remnant_azith.normalize_array(remnant_azith),
      extents.remnant_strength.normalize_array(remnant_strength),
    ],
    axis=-1,
  )
  applied_block = jnp.stack(
    [
      extents.applied_polar.normalize_array(applied_polar),
      extents.applied_azith.normalize_array(applied_azith),
      extents.applied_strength.normalize_array(applied_strength),
    ],
    axis=-1,
  )
  return jnp.concatenate([remnant_block.reshape(-1), applied_block.reshape(-1)])


def compute_initial_design(
  setup: CatheterSetup,
  extents: DesignExtents,
) -> np.ndarray:
  """Builds the start design of an axial half-strength catheter under a weak field.

  Every fiber carries `b_r = B_r,max t₀ / 2`, the conventional axially magnetized
  catheter. Every control carries `b_a = 0.05 B_a,max n₀`, where `n₀` is a unit vector
  normal to `t₀`. A small nonzero field is needed because at zero strength the applied
  angles have zero gradient, and a strength at its lower bound can stay there.

  Args:
    setup: `CatheterSetup` supplying the rod, the control count, and `t₀`.
    extents: `DesignExtents` used for the normalization.

  Returns:
    Array of shape `(num_design_var, 1)` of design variables in `[0, 1]`.
  """
  normal = np.cross(
    setup.entry_tangent, np.eye(3)[np.argmin(np.abs(setup.entry_tangent))]
  )
  remnant_mag_flux = np.broadcast_to(
    0.5 * max_remnant_flux * setup.entry_tangent,
    (setup.rod.fibers.num_elems, setup.rod.fibers.num_fibers, 3),
  )
  control_mag_flux = np.broadcast_to(
    0.05 * max_applied_flux * normal / np.linalg.norm(normal),
    (setup.field_interpolation.shape[1], 3),
  )
  design_vars = opt_vars_to_design_vars(
    remnant_mag_flux=jnp.asarray(remnant_mag_flux),
    control_mag_flux=jnp.asarray(control_mag_flux),
    extents=extents,
  )
  return np.asarray(design_vars).reshape(-1, 1)


def compute_node_positions(
  rod: _mesher.Rod,
  disp_history: jnp.ndarray,
) -> jnp.ndarray:
  """Computes the current node positions `x_n^k = X_n + u_n^k` at every step.

  Args:
    rod: `Rod` supplying the reference node coordinates `X_n`.
    disp_history: Array of shape `(num_steps, num_dofs)` [m, rad] of converged
      displacements and rotations at every step.

  Returns:
    Array of shape `(num_steps, num_nodes, 3)` [m] of node positions.
  """
  num_trans = rod.num_translational_dofs_per_node
  nodal = disp_history.reshape(disp_history.shape[0], rod.num_nodes, -1)
  return rod.node_coords[None, :, :] + nodal[..., :num_trans]


def compute_straight_positions(setup: CatheterSetup) -> np.ndarray:
  """Computes the node positions `p₀ + σ t₀` of a straight unactuated insertion.

  Args:
    setup: `CatheterSetup` supplying the rod, the depths, and `t₀`.

  Returns:
    Array of shape `(num_steps, num_nodes, 3)` [m] of node positions.
  """
  return np.asarray(setup.rod.node_coords)[None, :, :] + np.einsum(
    "k,d->kd", setup.insertion_depths, setup.entry_tangent
  )[:, None, :]

### Objective

In [ ]:
def compute_objective(
  design_vars: jnp.ndarray,
  extents: DesignExtents,
  setup: CatheterSetup,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
  """Computes the log relative path error over the insertion, its gradient, and state.

  The error sums the squared distance between every node and its target at every
  insertion step, `E = Σ_k Σ_n ‖x_n^k − p̂_n^k‖²`, and is divided by the straight
  insertion value `E₀`, so `J = log(E / E₀)` is zero for a straight insertion.
  Sheathed nodes sit on their targets exactly and contribute nothing. Each step is
  differentiated through its own implicit adjoint, since the elastic equilibrium at
  step `k` depends on the loads of step `k` only.

  Algorithm.
    1. Map the design vector to `b_r(e, f)` and `b_a(d_k)`.
    2. Solve the `K` insertion steps with the step-wise applied flux.
    3. Form the positions `x_n^k`, the error `x_n^k − p̂_n^k`, and `J = log(E / E₀)`.
    4. Differentiate `J` with respect to the design vector.

  Args:
    design_vars: Array of shape `(num_design_var,)` of design variables in `[0, 1]`.
    extents: `DesignExtents` supplying the physical range of each variable.
    setup: `CatheterSetup` supplying the problem, the targets, and `E₀`.

  Returns:
    A tuple `(objective, grad_objective, disp_history)`. `objective` has shape `(1,)`,
      `grad_objective` has shape `(num_design_var, 1)`, and `disp_history` has shape
      `(num_steps, num_dofs)` [m, rad].
  """
  shear_moduli = jnp.full(
    (setup.rod.fibers.num_elems, setup.rod.fibers.num_fibers), shear_modulus
  )

  def objective_fn(
    design_vars: jnp.ndarray,
  ) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Evaluates `J` and returns the displacement history as auxiliary."""
    remnant_mag_flux, applied_mag_flux = design_vars_to_opt_vars(
      design_vars=design_vars, extents=extents, setup=setup
    )
    _, disp_history = setup.problem.solve(
      u0=jnp.zeros(setup.rod.num_dofs),
      shear_moduli=shear_moduli,
      remnant_mag_flux=remnant_mag_flux,
      applied_mag_flux=applied_mag_flux,
    )
    error = (
      compute_node_positions(rod=setup.rod, disp_history=disp_history)
      - setup.target_positions
    )
    mismatch = jnp.einsum("knd,knd->", error, error)
    return jnp.log(mismatch / setup.straight_error), disp_history

  (objective, disp_history), grad_objective = jax.value_and_grad(
    objective_fn, has_aux=True
  )(design_vars)
  return objective.reshape(1), grad_objective.reshape(-1, 1), disp_history

### Gradient check through the insertion history

This is the first objective that differentiates the full load-step history rather than
the final state, so check one random direction once at the start design.

In [ ]:
objective_and_grad = jax.jit(
  ft.partial(compute_objective, extents=extents, setup=setup)
)
design_check = jnp.asarray(compute_initial_design(setup=setup, extents=extents))
design_check = design_check.reshape(-1)
direction = jax.random.normal(jax.random.PRNGKey(0), design_check.shape)
fd_step = 1e-5

_, grad_check, _ = objective_and_grad(design_check)
objective_plus, _, _ = objective_and_grad(design_check + fd_step * direction)
objective_minus, _, _ = objective_and_grad(design_check - fd_step * direction)
directional_fd = float(objective_plus[0] - objective_minus[0]) / (2.0 * fd_step)
directional_ad = float(jnp.einsum("n,n->", grad_check[:, 0], direction))
print(f"adjoint {directional_ad:.8e}  central FD {directional_fd:.8e}  "
      f"rel. err {abs(directional_ad - directional_fd) / abs(directional_fd):.2e}")

### Plotting

In [ ]:
def plot_convergence(history: dict[str, np.ndarray]) -> tuple[plt.Figure, plt.Axes]:
  """Plots the remaining error fraction `exp(J) = E / E₀` against the epoch.

  Args:
    history: Dictionary returned by `run_optimization`.

  Returns:
    The figure and its single axis.
  """
  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  ax.semilogy(history["epoch"], np.exp(history["objective"]), color="k", lw=1.2)
  ax.set_xlabel("Epoch")
  ax.set_ylabel(r"$E / E_0$")
  return fig, ax


def plot_insertion(
  setup: CatheterSetup,
  disp_history: np.ndarray,
  num_snapshots: int = 5,
  title: str | None = None,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the free catheter at evenly spaced insertion steps against the target path.

  Only nodes that have left the sheath, `σ ≥ 0`, are drawn. The box aspect follows the
  target extent, with every axis held open to at least a tenth of the longest one.

  Args:
    setup: `CatheterSetup` supplying the rod, the path coordinates, and the target.
    disp_history: Array of shape `(num_steps, num_dofs)` [m, rad] of converged states.
    num_snapshots: Number of insertion steps drawn, positive.
    title: Optional axis title, for example the epoch during monitoring.

  Returns:
    The figure and its single 3D axis.
  """
  positions = 1e3 * np.asarray(
    compute_node_positions(rod=setup.rod, disp_history=jnp.asarray(disp_history))
  )
  steps = np.linspace(0, setup.insertion_depths.shape[0] - 1, num_snapshots)
  colors = plt.cm.viridis(np.linspace(0.0, 1.0, num_snapshots))

  fig = plt.figure(figsize=(6.0, 5.0))
  ax = fig.add_subplot(projection="3d")
  ax.plot(*(1e3 * setup.target_points).T, color="k", ls="dashed", lw=1.0,
          label="Target path")
  for color, step in zip(colors, steps.round().astype(int)):
    free = setup.node_path_coords[step] >= 0.0
    ax.plot(*positions[step, free].T, color=color, lw=1.5,
            label=f"$d$ = {1e3 * setup.insertion_depths[step]:.1f} mm")
  extent = np.ptp(setup.target_points, axis=0)
  ax.set_box_aspect(np.maximum(extent, 0.1 * extent.max()))
  ax.set_xlabel(r"$x$ [mm]")
  ax.set_ylabel(r"$y$ [mm]")
  ax.set_zlabel(r"$z$ [mm]")
  ax.legend(fontsize=7, loc="upper left")
  if title is not None:
    ax.set_title(title)
  return fig, ax


def plot_tracking_error(
  setup: CatheterSetup,
  disp_history: np.ndarray,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the largest node deviation from the path at every insertion depth.

  The deviation `max_n ‖x_n^k − p̂_n^k‖` is shown for the design and for the straight
  unactuated insertion, which is the reference the objective is normalized by.

  Args:
    setup: `CatheterSetup` supplying the rod, the depths, and the targets.
    disp_history: Array of shape `(num_steps, num_dofs)` [m, rad] of converged states.

  Returns:
    The figure and its single axis.
  """
  targets = np.asarray(setup.target_positions)
  positions = np.asarray(
    compute_node_positions(rod=setup.rod, disp_history=jnp.asarray(disp_history))
  )
  deviation = np.linalg.norm(positions - targets, axis=-1).max(axis=1)
  straight = np.linalg.norm(
    compute_straight_positions(setup=setup) - targets, axis=-1
  ).max(axis=1)

  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  depth = 1e3 * setup.insertion_depths
  ax.plot(depth, 1e3 * deviation, color="k", lw=1.2, label="Magnetic")
  ax.plot(depth, 1e3 * straight, color="0.5", ls="dashed", lw=1.0,
          label="Straight insertion")
  ax.set_xlabel(r"Insertion depth $d$ [mm]")
  ax.set_ylabel(r"Largest deviation [mm]")
  ax.legend()
  return fig, ax


def plot_field_sequence(
  setup: CatheterSetup,
  applied_flux: np.ndarray,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the Cartesian components and the magnitude of `b_a` against depth.

  Args:
    setup: `CatheterSetup` supplying the insertion depths.
    applied_flux: Array of shape `(num_steps, 3)` [T] of the applied flux sequence.

  Returns:
    The figure and its single axis.
  """
  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  depth = 1e3 * setup.insertion_depths
  for component, label in enumerate((r"$b_{a,x}$", r"$b_{a,y}$", r"$b_{a,z}$")):
    ax.plot(depth, 1e3 * applied_flux[:, component], lw=1.2, label=label)
  ax.plot(depth, 1e3 * np.linalg.norm(applied_flux, axis=-1), color="k",
          ls="dashed", lw=1.0, label=r"$|b_a|$")
  ax.set_xlabel(r"Insertion depth $d$ [mm]")
  ax.set_ylabel(r"Applied flux [mT]")
  ax.legend(fontsize=8)
  return fig, ax


def plot_magnetization(
  setup: CatheterSetup,
  remnant_mag_flux: np.ndarray,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the material-frame section resultant of `b_r` against the distance from tip.

  Under a uniform field only the area-weighted section resultant
  `b̄_r = Σ_f A_f b_r,f / Σ_f A_f` enters the magnetic energy, so it is what is shown.
  The straight reference rod has one frame `{d₁, d₂, d₃}`, so the material components
  are `b̄_r · d_i`. Material beyond the path length from the tip never leaves the
  sheath, so its magnetization is irrelevant and it lies right of the dotted line.

  Args:
    setup: `CatheterSetup` supplying the rod and the path length.
    remnant_mag_flux: Array of shape `(num_elems, num_fibers, 3)` [T] of the remnant
      flux in the reference global frame.

  Returns:
    The figure and its single axis.
  """
  arc_coords = np.asarray(setup.rod.arc_coords)
  from_tip = 1e3 * (arc_coords[-1] - 0.5 * (arc_coords[:-1] + arc_coords[1:]))
  fiber_areas = np.asarray(setup.rod.fibers.areas)
  resultant = np.einsum("ef,efd->ed", fiber_areas, remnant_mag_flux) / np.sum(
    fiber_areas, axis=1, keepdims=True
  )
  material = np.einsum(
    "ed,rd->er", resultant, np.asarray(setup.rod.directors_0[0])
  )

  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  for component, label in enumerate((r"$b_r \cdot d_1$", r"$b_r \cdot d_2$",
                                     r"$b_r \cdot d_3$")):
    ax.plot(from_tip, 1e3 * material[:, component], lw=1.2, label=label)
  ax.axvline(1e3 * setup.insertion_depths[-1], color="0.5", ls="dotted", lw=1.0)
  ax.set_xlabel(r"Distance from tip [mm]")
  ax.set_ylabel(r"Remnant flux [mT]")
  ax.legend(fontsize=8)
  return fig, ax


def compute_wall_distance(points: np.ndarray, surface: pv.PolyData) -> np.ndarray:
  """Computes the unsigned distance from every point to the vessel surface.

  The distance is unsigned so that it reads as the clearance to the nearest wall
  both for a lumen surface and for a wall solid, provided the points lie in the lumen.

  Args:
    points: Array of shape `(..., 3)` [m] of query points.
    surface: Closed `pv.PolyData` of the vessel in m.

  Returns:
    Array of shape `(...)` [m] of distances to the surface.
  """
  query = pv.PolyData(points.reshape(-1, 3)).compute_implicit_distance(surface)
  return np.abs(np.asarray(query["implicit_distance"])).reshape(points.shape[:-1])


def plot_clearance(
  setup: CatheterSetup,
  clearance: np.ndarray,
  straight_clearance: np.ndarray,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the smallest wall clearance of the free catheter at every depth.

  The clearance is the wall distance of the centerline minus the outer radius, so a
  negative value means the catheter would touch the wall.

  Args:
    setup: `CatheterSetup` supplying the depths and the path coordinates.
    clearance: Array of shape `(num_steps, num_nodes)` [m] for the design.
    straight_clearance: Array of shape `(num_steps, num_nodes)` [m] for the straight
      insertion.

  Returns:
    The figure and its single axis.
  """
  free = setup.node_path_coords > 0.0
  depth = 1e3 * setup.insertion_depths
  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  ax.plot(depth, 1e3 * np.where(free, clearance, np.inf).min(axis=1), color="k",
          lw=1.2, label="Magnetic")
  ax.plot(depth, 1e3 * np.where(free, straight_clearance, np.inf).min(axis=1),
          color="0.5", ls="dashed", lw=1.0, label="Straight insertion")
  ax.axhline(0.0, color="r", lw=0.8)
  ax.set_xlabel(r"Insertion depth $d$ [mm]")
  ax.set_ylabel(r"Smallest wall clearance [mm]")
  ax.legend()
  return fig, ax

### MMA optimization

In [ ]:
def run_optimization(
  setup: CatheterSetup,
  extents: DesignExtents,
  max_iter: int = 200,
  min_epochs: int = 30,
  move_limit: float = 4e-2,
  save_interval: int = 5,
  monitor: bool = False,
) -> dict[str, np.ndarray]:
  """Runs MMA on the remnant flux angles and strengths and the applied flux sequence.

  The objective is bound to one setup and its extents and compiled once. The design
  is bounded by its box alone, so MMA receives one inactive dummy constraint, as in
  `mag_with_obstacle`. Converting every epoch's outputs to NumPy for MMA forces the
  device to finish, so the recorded wall time per epoch is exact, and its first entry
  includes compilation.

  Algorithm.
    1. Bind the setup and extents into the objective and compile it.
    2. Start from the axial half-strength remnant flux under a weak normal field.
    3. Evaluate the objective and its gradient and update the design with one MMA
       step, timing the whole epoch.
    4. Record the objective, the gradient norms of the remnant and applied blocks,
       MMA's KKT norm and design change, and the epoch time. Every `save_interval`
       epochs and at the last epoch, also record the design vector and the
       displacement history, and draw the insertion if `monitor` is set.
    5. Stop when MMA reports convergence after `min_epochs`, or at `max_iter`. MMA
       flags convergence when `‖x_k − x_{k−1}‖ < step_tol` or the KKT norm falls
       below `kkt_tol`, and either can fire on a tiny early step, so the printed
       design change and KKT norm show which one would have stopped the run.

  Args:
    setup: `CatheterSetup` of the case to optimize.
    extents: `DesignExtents` supplying the physical range of each variable.
    max_iter: Maximum number of MMA epochs, positive.
    min_epochs: Number of epochs before MMA convergence may end the run, positive.
    move_limit: MMA move limit per epoch, in `(0, 1]`. A value of `0.02` turns an
      azimuth by at most `0.04 π` per epoch.
    save_interval: Number of epochs between stored snapshots, positive.
    monitor: Whether to draw the insertion at every stored snapshot.

  Returns:
    A dictionary of arrays. `epoch`, `objective`, and `time_epoch` have shape
      `(num_epochs,)`, with times in seconds. `saved_epoch` has shape `(num_saved,)`,
      `design` has shape `(num_saved, num_design_var)`, and `disp` has shape
      `(num_saved, num_steps, num_dofs)`.
  """
  objective_and_grad = jax.jit(
    ft.partial(compute_objective, extents=extents, setup=setup)
  )
  initial_design = compute_initial_design(setup=setup, extents=extents)
  num_design_var = initial_design.shape[0]
  num_remnant_vars = 3 * setup.rod.fibers.num_elems * setup.rod.fibers.num_fibers
  mma_params = _mma.MMAParams(
    max_iter=max_iter,
    kkt_tol=1e-6,
    step_tol=1e-6,
    move_limit=move_limit,
    num_design_var=num_design_var,
    num_cons=1,
    lower_bound=np.zeros((num_design_var, 1)),
    upper_bound=np.ones((num_design_var, 1)),
  )
  mma_state = _mma.init_mma(init_design_var=initial_design, mma_params=mma_params)
  dummy_cons = np.full((1, 1), -1.0)
  dummy_grad_cons = np.zeros((1, num_design_var))
  keys = ("epoch", "objective", "time_epoch", "saved_epoch", "design", "disp")
  history = {key: [] for key in keys}

  while mma_state.epoch < max_iter and not (
    mma_state.is_converged and mma_state.epoch > min_epochs
  ):
    epoch = mma_state.epoch
    design_vars = jnp.asarray(mma_state.x).reshape(-1)

    epoch_start = time.perf_counter()
    objective, grad_objective, disp_history = objective_and_grad(design_vars)
    mma_state = _mma.update_mma(
      mma_state=mma_state,
      mma_params=mma_params,
      obj=np.asarray(objective, dtype=np.float64),
      grad_obj=np.asarray(grad_objective, dtype=np.float64),
      cons=dummy_cons,
      grad_cons=dummy_grad_cons,
    )
    time_epoch = time.perf_counter() - epoch_start

    history["epoch"].append(epoch)
    history["objective"].append(float(objective[0]))
    history["time_epoch"].append(time_epoch)
    grad = np.asarray(grad_objective)[:, 0]
    print(
      f"epoch {epoch:4d}  E/E0 {np.exp(float(objective[0])):.3e}  "
      f"|dJ/dbr| {np.linalg.norm(grad[:num_remnant_vars]):.2e}  "
      f"|dJ/dba| {np.linalg.norm(grad[num_remnant_vars:]):.2e}  "
      f"kkt {float(mma_state.kkt_norm):.2e}  "
      f"dx {float(mma_state.change_design_var):.2e}  {time_epoch:.2f} s"
    )

    is_last = mma_state.epoch >= max_iter or (
      mma_state.is_converged and mma_state.epoch > min_epochs
    )
    if epoch % save_interval == 0 or is_last:
      history["saved_epoch"].append(epoch)
      history["design"].append(np.asarray(design_vars))
      history["disp"].append(np.asarray(disp_history))
      if monitor:
        plot_insertion(setup=setup, disp_history=history["disp"][-1],
                       title=f"epoch {epoch}")
        plt.show()

  return {key: np.asarray(value) for key, value in history.items()}

### Run

In [ ]:
history = run_optimization(setup=setup, extents=extents, monitor=True)
np.savez(results_dir / "baseline.npz", **history, **baseline_settings)

### Results

In [ ]:
remnant_mag_flux, applied_mag_flux = (
  np.asarray(field)
  for field in design_vars_to_opt_vars(
    design_vars=jnp.asarray(history["design"][-1]), extents=extents, setup=setup
  )
)
figures = {
  "convergence": plot_convergence(history=history),
  "insertion": plot_insertion(setup=setup, disp_history=history["disp"][-1]),
  "tracking_error": plot_tracking_error(
    setup=setup, disp_history=history["disp"][-1]
  ),
  "field_sequence": plot_field_sequence(
    setup=setup, applied_flux=applied_mag_flux
  ),
  "magnetization": plot_magnetization(
    setup=setup, remnant_mag_flux=remnant_mag_flux
  ),
}
for name, (fig, _) in figures.items():
  fig.savefig(results_dir / f"baseline_{name}.pdf", bbox_inches="tight")
plt.show()

### Wall clearance from the STL, post-processing only

The wall never enters the solve. The STL is shifted by the same origin as the csv.
The first print is the distance from the target path to the wall, which should read as
the lumen radius. If it does not, the STL units or frame differ from the csv, and
`stl_length_scale` needs fixing before the clearance plot means anything.

In [ ]:
surface = pv.read(stl_file)
surface.points = stl_length_scale * surface.points - setup.origin

lumen_radius = compute_wall_distance(points=setup.target_points, surface=surface)
print(f"path to wall, min {1e3 * lumen_radius.min():.2f} mm, "
      f"mean {1e3 * lumen_radius.mean():.2f} mm, "
      f"catheter radius {1e3 * outer_radius:.2f} mm")

positions = np.asarray(
  compute_node_positions(rod=setup.rod, disp_history=jnp.asarray(history["disp"][-1]))
)
fig, ax = plot_clearance(
  setup=setup,
  clearance=compute_wall_distance(points=positions, surface=surface) - outer_radius,
  straight_clearance=(
    compute_wall_distance(points=compute_straight_positions(setup=setup),
                          surface=surface)
    - outer_radius
  ),
)
fig.savefig(results_dir / "baseline_clearance.pdf", bbox_inches="tight")
plt.show()